# Chuyển đổi định dạng dữ liệu từ COCO sang YOLO
Notebook này thực hiện bước chuẩn hóa dữ liệu trước khi train model. Dữ liệu gốc của Zalo AI lưu theo chuẩn COCO, còn YOLO/RT-DETR cần file `.txt` riêng cho từng ảnh.

Mục tiêu:
- đọc JSON gốc từ Hugging Face hoặc local path,
- chuyển bbox từ dạng COCO `[x, y, width, height]` sang dạng YOLO `[class_id, cx, cy, w, h]` chuẩn hóa theo tỷ lệ ảnh,
- lưu ra thư mục mới: `data/yolo_format/images/` và `data/yolo_format/labels/`.

In [3]:
import json
import os
from pathlib import Path

# Cài đặt gói hỗ trợ tải dataset từ Hugging Face bằng hf_transfer nếu chưa có
# Đây là cách đơn giản và tương thích với môi trường phục vụ đồ án.
try:
    import hf_transfer  # noqa: F401
except ModuleNotFoundError:
    os.system('pip install hf_transfer')

from huggingface_hub import hf_hub_download

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN / REPO HF
# ==========================================
# Nếu có dataset trên Hugging Face, ta sẽ tải về theo repo_id dưới đây.
# Nếu không, hãy thay bằng đường dẫn local đến file JSON gốc.
REPO_ID = ""
FILE_NAME = "train_traffic_sign_dataset.json"
LOCAL_JSON_PATH = r"G:\HocKi9\Học Thống Kê\topic_final\final\archive\za_traffic_2020\traffic_train\train_traffic_sign_dataset.json"

# Tạo thư mục output riêng cho format YOLO, tránh làm rác vào thư mục gốc.
project_root = Path.cwd().resolve().parent
output_root = project_root / "data" / "yolo_format"
images_out = output_root / "images"
labels_out = output_root / "labels"
images_out.mkdir(parents=True, exist_ok=True)
labels_out.mkdir(parents=True, exist_ok=True)

# Khởi tạo biến lưu dữ liệu JSON gốc
json_path = None

# Trường hợp 1: tải trực tiếp từ Hugging Face bằng hf_transfer
if REPO_ID:
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    json_path = hf_hub_download(repo_id=REPO_ID, filename=FILE_NAME, repo_type="dataset")

# Trường hợp 2: đọc từ file local nếu chưa có repo_id
if json_path is None and os.path.exists(LOCAL_JSON_PATH):
    json_path = LOCAL_JSON_PATH

if json_path is None:
    raise FileNotFoundError("Không tìm thấy JSON gốc. Hãy điền REPO_ID hoặc cập nhật LOCAL_JSON_PATH.")

with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Đã đọc dữ liệu từ: {json_path}")
print(f"Tổng số ảnh: {len(data['images'])}")
print(f"Tổng số annotation: {len(data['annotations'])}")
print(f"Output YOLO sẽ lưu ở: {output_root}")

Đã đọc dữ liệu từ: G:\HocKi9\Học Thống Kê\topic_final\final\archive\za_traffic_2020\traffic_train\train_traffic_sign_dataset.json
Tổng số ảnh: 4500
Tổng số annotation: 11000
Output YOLO sẽ lưu ở: G:\HocKi9\Học Thống Kê\topic_final\final\Object-Detection-Application\Traffic-Sign-Detection-ZaloAI\data\yolo_format


In [4]:
# Chuyển đổi bbox từ định dạng COCO sang định dạng YOLO chuẩn
# COCO: [x, y, width, height] theo pixel ảnh gốc
# YOLO: [class_id, x_center, y_center, width, height] theo tỷ lệ 0..1

# Đọc lại dữ liệu để đảm bảo mọi thứ còn nguyên cho phần chuyển đổi
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

image_map = {img['id']: img for img in data['images']}
annotations_by_image = {}
for ann in data['annotations']:
    annotations_by_image.setdefault(ann['image_id'], []).append(ann)

# Lưu ý: cần tạo label file cho từng ảnh một, vì YOLO yêu cầu 1 file txt tương ứng với một ảnh
# Tên file ảnh gốc và file nhãn phải trùng tên nhưng khác extension.
for img in data['images']:
    img_id = img['id']
    file_name = img['file_name']
    img_width = img['width']
    img_height = img['height']

    # Copy ảnh gốc sang thư mục images của YOLO format
    source_path = Path(LOCAL_JSON_PATH).parent / 'images' / file_name
    if source_path.exists():
        target_path = images_out / file_name
        target_path.write_bytes(source_path.read_bytes())
    else:
        # Nếu dùng Hugging Face, nó sẽ download ở local trước khi gọi file này.
        # Ở thời điểm này, file có thể chưa có trong thư mục images gốc.
        # Cách làm đơn giản là bỏ qua việc copy nếu dataset được mount từ repo khác.
        print(f"Bỏ qua copy ảnh: {file_name} vì không tìm thấy trong folder local.")

    label_lines = []
    for ann in annotations_by_image.get(img_id, []):
        class_id = ann['category_id']
        x, y, w, h = ann['bbox']

        # Chuyển bbox COCO sang YOLO theo tỷ lệ chuẩn hóa
        x_center = (x + w / 2) / img_width
        y_center = (y + h / 2) / img_height
        norm_w = w / img_width
        norm_h = h / img_height

        # Giữ nguyên class_id từ COCO, nhưng nếu model yêu cầu class index bắt đầu từ 0,
        # ta có thể cộng trừ ở đây tùy bộ dữ liệu.
        label_lines.append(f"{class_id - 1} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")

    label_path = labels_out / (Path(file_name).stem + '.txt')
    with open(label_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(label_lines))

    if len(label_lines) == 0:
        # File nhãn rỗng vẫn nên được giữ lại để model không bị lỗi khi quét các ảnh không có vật thể.
        label_path.write_text('', encoding='utf-8')

print("Đã chuyển xong dữ liệu COCO sang YOLO format.")
print(f"Số file ảnh được copy: {len(list(images_out.iterdir()))}")
print(f"Số file label được tạo: {len(list(labels_out.iterdir()))}")
print("Cấu trúc output:")
for p in sorted(output_root.rglob('*'))[:10]:
    print(' -', p.relative_to(project_root))

Đã chuyển xong dữ liệu COCO sang YOLO format.
Số file ảnh được copy: 4500
Số file label được tạo: 4500
Cấu trúc output:
 - data\yolo_format\images
 - data\yolo_format\images\1000.png
 - data\yolo_format\images\10002.png
 - data\yolo_format\images\10005.png
 - data\yolo_format\images\10006.png
 - data\yolo_format\images\10008.png
 - data\yolo_format\images\10011.png
 - data\yolo_format\images\10014.png
 - data\yolo_format\images\10016.png
 - data\yolo_format\images\10018.png
